In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os
from tqdm.auto import tqdm
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from scipy.spatial.distance import pdist, squareform

# --- 1. SETUP AND CONFIGURATION ---
class Config:
    # FIXED: Use correct path to combined BFM data
    RAW_CSV_PATH = "Cleaned and combined/bfm_data.csv"    
    # --- NEW: Define a prefix for saving models ---
    # We will save 6 different models
    MODEL_SAVE_PREFIX = 'bfm_experiment_model' 
    
    # --- NEW: Define all subjects and environments ---
    ALL_SUBJECTS = ['abel', 'matthew', 'ivan', 'kenny', 'collin']
    ALL_ENVIRONMENTS = ['nofoil', 'foil', 'open']
    
    # --- NEW: Define test subject for Part 1 ---
    TEST_SUBJECT_PART1 = ['matthew']
    
    # --- NEW: Define train/test split ratio for Part 2 ---
    SESSION_TRAIN_RATIO = 0.8
    SEED = 42 # For reproducible session splits
    
    ACTIVITIES = ['standing', 'walking'] 
    ACTIVITY_MAP = {name: i for i, name in enumerate(ACTIVITIES)}
    NUM_CLASSES = len(ACTIVITIES)
    IMAGE_SIZE = (128, 128)
    BATCH_SIZE = 32
    EPOCHS = 15 
    LEARNING_RATE = 0.001
    WINDOW_SIZE = 50

config = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set seed for reproducibility of random splits in Part 2
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)

# --- 2. MODEL ARCHITECTURE (Unchanged) ---
class ResNetWithFeatures(nn.Module):
    def __init__(self, num_classes):
        super(ResNetWithFeatures, self).__init__()
        # Using pre-trained weights=None as in the original script
        resnet = models.resnet18(weights=None) 
        # Adapt to 1-channel (grayscale) input
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        num_ftrs = resnet.fc.in_features
        self.classifier = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        features = self.feature_extractor(x)
        features_flat = torch.flatten(features, 1)
        logits = self.classifier(features_flat)
        # We return features_flat for consistency, but won't use it in supervised loss
        return logits, features_flat

# --- 3. DATASET CLASS (Unchanged) ---
class OnTheFlyRPDataset(Dataset):
    def __init__(self, dataframe, mag_cols, activity_map, window_size, transform=None):
        self.df = dataframe
        self.mag_cols = mag_cols
        self.activity_map = activity_map
        self.window_size = window_size
        self.transform = transform
        
        self.samples = []
        # Group by session_id first to get the label
        session_groups = self.df.groupby('session_id').first()
        for session_id, session_row in session_groups.iterrows():
            label = self.activity_map[session_row['activity']]
            for subcarrier in self.mag_cols:
                # Each (session, subcarrier) is one sample
                self.samples.append((session_id, subcarrier, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        session_id, subcarrier_name, label = self.samples[idx]
        
        # Get all time-series data for this session_id
        time_series_raw = self.df[self.df['session_id'] == session_id][subcarrier_name].values
        
        if len(time_series_raw) > 1:
            series_pd = pd.Series(time_series_raw)
            # Apply moving window normalization
            rolling_mean = series_pd.rolling(window=self.window_size, min_periods=1).mean()
            rolling_std = series_pd.rolling(window=self.window_size, min_periods=1).std()
            normalized_ts = (series_pd - rolling_mean) / (rolling_std + 1e-6)
            normalized_ts.replace([np.inf, -np.inf], np.nan, inplace=True)
            normalized_ts.fillna(0, inplace=True) # Fill NaNs (e.g., from std=0)
            time_series = normalized_ts.values
        else:
            time_series = time_series_raw

        # Create Recurrence Plot
        reshaped_ts = time_series.reshape(-1, 1)
        rp_matrix = squareform(pdist(reshaped_ts, 'euclidean'))
        
        # Convert to tensor and apply transforms
        image_tensor = torch.from_numpy(rp_matrix).float().unsqueeze(0) # Add channel dim
        if self.transform:
            image_tensor = self.transform(image_tensor)
            
        return image_tensor, label

# --- 4. LOSS FUNCTION (REMOVED) ---
# SubdomainAlignmentLoss is no longer needed. We will use nn.CrossEntropyLoss directly.


# --- 5. TRAINING AND VALIDATION FUNCTIONS (MODIFIED) ---

def train_one_epoch_supervised(model, train_loader, optimizer, loss_fn):
    """
    NEW: Standard supervised training loop.
    """
    model.train()
    total_loss = 0
    
    for data, labels in tqdm(train_loader, desc="Training", leave=False):
        data, labels = data.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Model returns (logits, features)
        logits, _ = model(data) 
        
        # Calculate standard classification loss
        loss = loss_fn(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(train_loader)

def validate(model, data_loader, set_name="Test"):
    """
    (Unchanged) Standard validation loop.
    """
    model.eval()
    all_labels, all_predictions = [], []
    
    with torch.no_grad():
        for data, labels in tqdm(data_loader, desc=f"Evaluating on {set_name} Set", leave=False):
            data, labels = data.to(device), labels.to(device)
            logits, _ = model(data)
            _, predicted = torch.max(logits.data, 1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
            
    accuracy = 100 * np.sum(np.array(all_labels) == np.array(all_predictions)) / len(all_labels)
    return accuracy, all_labels, all_predictions
    
def plot_confusion_matrix(true_labels, pred_labels, class_names, title="Confusion Matrix"):
    """
    (Slightly modified to accept a title)
    """
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.show()
    
# --- 6. MAIN SCRIPT (COMPLETELY REVISED) ---
if __name__ == '__main__':
    print("\n--- Loading and Preparing Data ---")
    
    # Check if the file exists before reading
    if not os.path.exists(config.RAW_CSV_PATH):
        print(f"Error: Data file not found at {config.RAW_CSV_PATH}")
        print("Please check the file path in the Config class.")
    else:
        df = pd.read_csv(config.RAW_CSV_PATH)
        df['session_id'] = df['session_id'].astype(str).str.strip()
        mag_cols = [col for col in df.columns if 'SCIDX_' in col and 'Ratio_Mag' in col]
        print(f"Data loaded successfully. Found {len(df)} rows and {len(mag_cols)} subcarrier columns.")

        # Define transformations
        train_transform = transforms.Compose([
            transforms.Resize(config.IMAGE_SIZE),
            transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
        ])
        test_transform = transforms.Compose([
            transforms.Resize(config.IMAGE_SIZE)
        ])

        # ======================================================================
        # PART 1: LEAVE-ONE-SUBJECT-OUT (Per Environment)
        # Train on 4 subjects, Test on 1 subject, all from the same environment
        # ======================================================================
        print("\n" + "="*60)
        print("=== PART 1: LEAVE-ONE-SUBJECT-OUT (Per Environment) ===")
        print("="*60 + "\n")

        # Define the train/test split for Part 1
        test_subjects_p1 = config.TEST_SUBJECT_PART1
        train_subjects_p1 = [s for s in config.ALL_SUBJECTS if s not in test_subjects_p1]
        print(f"Part 1 Setup: Training on {train_subjects_p1}, Testing on {test_subjects_p1}")

        for env in config.ALL_ENVIRONMENTS:
            print(f"\n--- Starting Part 1 Experiment: Environment '{env}' ---")
            
            # 1. Filter data for the environment
            env_df = df[df['environment'] == env]
            train_df = env_df[env_df['subject'].isin(train_subjects_p1)]
            test_df = env_df[env_df['subject'].isin(test_subjects_p1)]
            
            print(f"Found {len(train_df)} training rows, {len(test_df)} test rows.")
            if len(train_df) == 0 or len(test_df) == 0:
                print(f"Skipping {env}: Not enough data for train/test split.")
                continue

            # 2. Create Datasets and DataLoaders
            # FIXED: Changed num_workers to 0 to avoid multiprocessing issues in Jupyter
            train_dataset = OnTheFlyRPDataset(train_df, mag_cols, config.ACTIVITY_MAP, config.WINDOW_SIZE, transform=train_transform)
            test_dataset = OnTheFlyRPDataset(test_df, mag_cols, config.ACTIVITY_MAP, config.WINDOW_SIZE, transform=test_transform)
            
            train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0)
            test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)
            
            # 3. Initialize Model, Optimizer, Loss for this experiment
            model = ResNetWithFeatures(num_classes=config.NUM_CLASSES).to(device)
            optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
            classification_loss_fn = nn.CrossEntropyLoss() # Use standard CE Loss

            print(f"\nStarting training for {config.EPOCHS} epochs...")
            for epoch in range(config.EPOCHS):
                avg_loss = train_one_epoch_supervised(model, train_loader, optimizer, classification_loss_fn)
                print(f"Epoch {epoch+1}/{config.EPOCHS} - Avg. Training Loss: {avg_loss:.4f}")
            
            print("\n--- Training Finished ---")
            
            # 4. Evaluate
            print(f"\nRunning final evaluation on test set (Subject: {test_subjects_p1}, Env: {env})...")
            test_accuracy, true_labels, pred_labels = validate(model, test_loader, set_name="Test")
            print(f"\nFINAL TEST ACCURACY (Part 1, Env: {env}): {test_accuracy:.2f}%")
            
            print("\nGenerating confusion matrix...")
            cm_title = f"Part 1: Env='{env}', Test Subject='{test_subjects_p1[0]}'"
            plot_confusion_matrix(true_labels, pred_labels, config.ACTIVITIES, title=cm_title)
            
            # 5. Save Model
            save_path = f"{config.MODEL_SAVE_PREFIX}_part1_env_{env}.pth"
            print(f"\n--- Saving Model to {save_path} ---")
            torch.save(model.state_dict(), save_path)


        # ======================================================================
        # PART 2: 80/20 SESSION-SPLIT (Per Environment)
        # Train on 80% of sessions from all subjects, Test on 20%
        # ======================================================================
        print("\n" + "="*60)
        print(f"=== PART 2: {config.SESSION_TRAIN_RATIO*100:.0f}/{100-config.SESSION_TRAIN_RATIO*100:.0f} SESSION-SPLIT (Per Environment) ===")
        print("="*60 + "\n")
        
        for env in config.ALL_ENVIRONMENTS:
            print(f"\n--- Starting Part 2 Experiment: Environment '{env}' ---")
            
            # 1. Filter data for the environment
            env_df = df[df['environment'] == env]
            
            # 2. Create 80/20 session split
            train_session_ids = []
            test_session_ids = []
            
            print(f"Creating {config.SESSION_TRAIN_RATIO*100:.0f}/{100-config.SESSION_TRAIN_RATIO*100:.0f} session split across all subjects...")
            for subject in config.ALL_SUBJECTS:
                for activity in config.ACTIVITIES:
                    # Find all unique sessions for this subject/activity/env
                    subject_activity_sessions = env_df[
                        (env_df['subject'] == subject) & 
                        (env_df['activity'] == activity)
                    ]['session_id'].unique()
                    
                    if len(subject_activity_sessions) == 0:
                        # print(f"Warning: No sessions found for {subject}/{activity} in {env}. Skipping.")
                        continue
                    
                    # Shuffle and split
                    np.random.shuffle(subject_activity_sessions) # In-place shuffle
                    split_idx = int(len(subject_activity_sessions) * config.SESSION_TRAIN_RATIO)
                    
                    # Handle very small numbers (e.g., if only 1 session, put in train)
                    if split_idx == 0 and len(subject_activity_sessions) > 0:
                        split_idx = 1
                    
                    train_session_ids.extend(subject_activity_sessions[:split_idx])
                    test_session_ids.extend(subject_activity_sessions[split_idx:])

            # Create DataFrames from the session ID lists
            train_df = env_df[env_df['session_id'].isin(train_session_ids)]
            test_df = env_df[env_df['session_id'].isin(test_session_ids)]

            print(f"Found {len(train_df)} training rows (from {len(np.unique(train_session_ids))} unique sessions)")
            print(f"Found {len(test_df)} test rows (from {len(np.unique(test_session_ids))} unique sessions)")

            # Check for leak (sets should be disjoint)
            train_sessions_set = set(train_session_ids)
            test_sessions_set = set(test_session_ids)
            if not train_sessions_set.isdisjoint(test_sessions_set):
                print("!!! WARNING: DATA LEAK DETECTED - SESSIONS ARE IN BOTH TRAIN AND TEST !!!")
            else:
                print("Session split successful. No leakage detected.")

            if len(train_df) == 0 or len(test_df) == 0:
                print(f"Skipping {env}: Not enough data for train/test split.")
                continue

            # 3. Create Datasets and DataLoaders
            # FIXED: Changed num_workers to 0 to avoid multiprocessing issues in Jupyter
            train_dataset = OnTheFlyRPDataset(train_df, mag_cols, config.ACTIVITY_MAP, config.WINDOW_SIZE, transform=train_transform)
            test_dataset = OnTheFlyRPDataset(test_df, mag_cols, config.ACTIVITY_MAP, config.WINDOW_SIZE, transform=test_transform)
            
            train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0)
            test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)
            
            # 4. Initialize Model, Optimizer, Loss
            model = ResNetWithFeatures(num_classes=config.NUM_CLASSES).to(device)
            optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
            classification_loss_fn = nn.CrossEntropyLoss()

            print(f"\nStarting training for {config.EPOCHS} epochs...")
            for epoch in range(config.EPOCHS):
                avg_loss = train_one_epoch_supervised(model, train_loader, optimizer, classification_loss_fn)
                print(f"Epoch {epoch+1}/{config.EPOCHS} - Avg. Training Loss: {avg_loss:.4f}")
            
            print("\n--- Training Finished ---")
            
            # 5. Evaluate
            print(f"\nRunning final evaluation on 20% test set (Env: {env})...")
            test_accuracy, true_labels, pred_labels = validate(model, test_loader, set_name="Test")
            print(f"\nFINAL TEST ACCURACY (Part 2, Env: {env}): {test_accuracy:.2f}%")
            
            print("\nGenerating confusion matrix...")
            cm_title = f"Part 2: Env='{env}', 80/20 Session Split"
            plot_confusion_matrix(true_labels, pred_labels, config.ACTIVITIES, title=cm_title)
            
            # 6. Save Model
            save_path = f"{config.MODEL_SAVE_PREFIX}_part2_env_{env}.pth"
            print(f"\n--- Saving Model to {save_path} ---")
            torch.save(model.state_dict(), save_path)
            
        print("\n--- All 6 experiments finished. ---")

## Model saving and Testing to whole new dataset 

In [ ]:
# In a new script or cell:

# 1. You MUST define the model architecture class again
#    (Just copy the ResNetWithFeatures class definition here)
# class ResNetWithFeatures(nn.Module):
    # ... (paste the full class definition from the training script)

# 2. Define your configuration again
#    (You need NUM_CLASSES and MODEL_SAVE_PATH)
num_classes = 2 # Or get from a config object
model_path = '/kaggle/working/wisda_resnet_model.pth'

# 3. Instantiate the model and load the saved weights
#    Make sure to move the model to the correct device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetWithFeatures(num_classes=num_classes).to(device)
model.load_state_dict(torch.load(model_path))

# 4. VERY IMPORTANT: Set the model to evaluation mode
#    This disables layers like Dropout and BatchNorm that behave differently during training
model.eval()

print("Model loaded successfully and set to evaluation mode.")

# Now your 'model' is ready to make predictions on new data!
# For example, to predict on a single sample:
# with torch.no_grad():
#     # 'sample_tensor' would be a new, preprocessed RP image
#     logits, features = model(sample_tensor.to(device))
#     _, prediction = torch.max(logits, 1)
#     print(f"Predicted class index: {prediction.item()}")

In [ ]:
def evaluate_model_performance(model, test_df, config, device):
    """
    Evaluates a trained model on a given DataFrame.

    Args:
        model (nn.Module): The trained PyTorch model.
        test_df (pd.DataFrame): The DataFrame containing the test data.
        config (Config): The configuration object.
        device (torch.device): The device to run evaluation on (CPU or CUDA).
    """
    print(f"\n--- Starting Evaluation ---")
    
    # 1. Set model to evaluation mode
    model.eval()
    
    # 2. Prepare the dataset and dataloader
    mag_cols = [col for col in test_df.columns if 'SCIDX_' in col and 'Ratio_Mag' in col]
    test_transform = transforms.Compose([transforms.Resize(config.IMAGE_SIZE)])
    
    test_dataset = OnTheFlyRPDataset(
        dataframe=test_df,
        mag_cols=mag_cols,
        activity_map=config.ACTIVITY_MAP,
        window_size=config.WINDOW_SIZE,
        transform=test_transform
    )
    
    if len(test_dataset) == 0:
        print("Error: The provided DataFrame resulted in an empty dataset. No evaluation can be performed.")
        return

    # FIXED: Changed num_workers to 0 to avoid multiprocessing issues in Jupyter
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 3. Perform validation
    all_labels, all_predictions = [], []
    with torch.no_grad():
        for data, labels in tqdm(test_loader, desc="Evaluating"):
            data, labels = data.to(device), labels.to(device)
            logits, _ = model(data)
            _, predicted = torch.max(logits.data, 1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
            
    # 4. Calculate accuracy
    accuracy = 100 * np.sum(np.array(all_labels) == np.array(all_predictions)) / len(all_labels)
    print(f"\nFinal Test Accuracy: {accuracy:.2f}%")
    
    # 5. Generate and display confusion matrix
    print("\nGenerating confusion matrix...")
    cm = confusion_matrix(all_labels, all_predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=config.ACTIVITIES, yticklabels=config.ACTIVITIES)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix on Test Set')
    plt.show()
    
    return accuracy

In [ ]:
evaluate_model_performance(model, df, config, 'cuda')